# MLP Name Generator

**Barkeep Protocol — Act 1, Week 2, Days 3–4**

Bengio et al. (2003) applied to character-level name generation.
Architecture: embedding table → concat context → tanh hidden layer → logits → cross-entropy.

In [1]:
import torch
import random
import numpy as np
import matplotlib.pyplot as plt

## Data

In [ ]:
words = open("Names.txt", "r").read().splitlines()

In [ ]:
letters = sorted(set("".join(words)))

ltr_to_idx = {ltr: idx + 1 for idx, ltr in enumerate(letters)}
ltr_to_idx["."] = 0

idx_to_ltr = {idx: ltr for ltr, idx in ltr_to_idx.items()}

VOCAB_SIZE = len(ltr_to_idx)
print(f"Vocabulary size: {VOCAB_SIZE} | Words: {len(words):,}")

In [ ]:
block_size = 5

def build_dataset(words):
    Xs, Ys = [], []

    for word in words:
        context = [0] * block_size

        for ltr in word + '.':
            idx = ltr_to_idx[ltr]
            Xs.append(context)
            Ys.append(idx)
            context = context[1:] + [idx]

    return (
        torch.tensor(Xs, dtype=torch.int),
        torch.tensor(Ys, dtype=torch.long),
    )

In [ ]:
random.seed(42)
random.shuffle(words)

n1 = int(0.8 * len(words))
n2 = int(0.9 * len(words))

trn_inputs,  trn_outputs  = build_dataset(words[:n1])
val_inputs,  val_outputs  = build_dataset(words[n1:n2])
test_inputs, test_outputs = build_dataset(words[n2:])

for split, t in [("train", trn_inputs), ("val", val_inputs), ("test", test_inputs)]:
    print(f"{split:>5}: {t.shape[0]:>7,} examples")

## Model

In [ ]:
sampler = torch.Generator().manual_seed(42)

embedding_dimensionality = 50
d1 = block_size * embedding_dimensionality
d2 = 500

embedding_matrix = torch.randn((VOCAB_SIZE, embedding_dimensionality), generator=sampler) * 0.1
weights_1        = torch.randn((d1, d2),                               generator=sampler) * 0.1
bias_1           = torch.zeros(d2)
weights_2        = torch.randn((d2, VOCAB_SIZE),                       generator=sampler) * 0.1
bias_2           = torch.zeros(VOCAB_SIZE)

parameters = [embedding_matrix, weights_1, bias_1, weights_2, bias_2]
for p in parameters:
    p.requires_grad = True

print(f"Parameters: {sum(p.nelement() for p in parameters):,}")

In [ ]:
def embed_inputs(inputs):
    return embedding_matrix[inputs]

## Training

In [ ]:
total_steps = 500000
lr_start    = 0.5
lr_end      = 0.01

lr_per_itrn   = []
loss_per_itrn = []
val_losses    = []
val_steps     = []

for turn in range(total_steps):
    idx = torch.randint(0, trn_inputs.shape[0], (1024,))

    embedded_inputs = embed_inputs(trn_inputs[idx])
    hidden_layer    = torch.tanh(embedded_inputs.view(-1, d1) @ weights_1 + bias_1)
    logits          = hidden_layer @ weights_2 + bias_2
    loss            = torch.nn.functional.cross_entropy(logits, trn_outputs[idx])

    for p in parameters:
        p.grad = None
    loss.backward()

    lr = lr_start + (lr_end - lr_start) * (turn / total_steps)
    for p in parameters:
        p.data += -lr * p.grad

    lr_per_itrn.append(lr)
    loss_per_itrn.append(loss.item())

    if (turn + 1) % 100000 == 0:
        with torch.no_grad():
            emb      = embed_inputs(val_inputs)
            h        = torch.tanh(emb.view(-1, d1) @ weights_1 + bias_1)
            val_loss = torch.nn.functional.cross_entropy(h @ weights_2 + bias_2, val_outputs)
        val_losses.append(val_loss.item())
        val_steps.append(turn + 1)
        print(f"  step {turn + 1:>7,} | val {val_loss.item():.4f} | lr {lr:.4f}")

## Visualisation

In [ ]:
import sys
sys.path.insert(0, '..')
import barkeep_style as bks
bks.apply_style()

In [ ]:
# Training loss curve
fig, ax = plt.subplots(figsize=(10, 5))

steps = range(len(loss_per_itrn))
ax.plot(steps, loss_per_itrn, color=bks.COLORS["amber"], alpha=0.08, linewidth=0.5)

# Windowed moving average — O(n) via prefix sums
window = 1000
arr    = np.asarray(loss_per_itrn)
csum   = np.concatenate(([0.0], np.cumsum(arr)))
idx    = np.arange(len(arr))
lo     = np.maximum(0, idx - window)
smoothed = (csum[idx + 1] - csum[lo]) / (idx + 1 - lo)
ax.plot(steps, smoothed, color=bks.COLORS["amber"], linewidth=1.5, label="Train loss (smoothed)")

ax.scatter(val_steps, val_losses, color=bks.COLORS["red"], s=40, zorder=5,
           marker="D", label="Val loss")
for s, v in zip(val_steps, val_losses):
    ax.text(s, v + 0.03, f"{v:.3f}", ha="center", fontsize=9, color=bks.COLORS["red"])

ax.set_xlabel("Step")
ax.set_ylabel("Loss")
ax.set_title("Training Loss — MLP Name Generator")
ax.legend()
plt.show()

In [ ]:
# Learning rate schedule
fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(range(len(lr_per_itrn)), lr_per_itrn, color=bks.COLORS["teal"], linewidth=1.5)
ax.set_xlabel("Step")
ax.set_ylabel("Learning Rate")
ax.set_title("Learning Rate Schedule")
plt.show()

In [ ]:
# Train vs val loss at checkpoints
trn_at_checkpoints = []
for s in val_steps:
    w = max(0, s - 5000)
    trn_at_checkpoints.append(sum(loss_per_itrn[w:s]) / (s - w))

fig, ax = plt.subplots(figsize=(8, 4))
x_pos, bar_w = range(len(val_steps)), 0.35
ax.bar([p - bar_w / 2 for p in x_pos], trn_at_checkpoints, bar_w, color=bks.COLORS["amber"], label="Train")
ax.bar([p + bar_w / 2 for p in x_pos], val_losses,         bar_w, color=bks.COLORS["red"],   label="Val")
ax.set_xticks(list(x_pos))
ax.set_xticklabels([f"{s // 1000}K" for s in val_steps])
ax.set_xlabel("Step")
ax.set_ylabel("Loss")
ax.set_title("Train vs Val Loss at Checkpoints")
ax.legend()
plt.show()

## Evaluation

In [ ]:
with torch.no_grad():
    emb     = embed_inputs(trn_inputs)
    h       = torch.tanh(emb.view(-1, d1) @ weights_1 + bias_1)
    trn_loss = torch.nn.functional.cross_entropy(h @ weights_2 + bias_2, trn_outputs)

print(f"Train: {trn_loss.item():.4f} | Val: {val_losses[-1]:.4f}")

## Sampling

In [ ]:
for _ in range(20):
    context = [0] * block_size
    name    = []

    while True:
        emb    = embedding_matrix[torch.tensor([context])]
        h      = torch.tanh(emb.view(1, -1) @ weights_1 + bias_1)
        logits = h @ weights_2 + bias_2
        probs  = torch.softmax(logits, dim=1)
        idx    = torch.multinomial(probs, num_samples=1).item()

        if idx == 0:
            break
        name.append(idx_to_ltr[idx])
        context = context[1:] + [idx]

    print("".join(name))